In [ ]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

LOCAL_TRL_PARENT = "/root/autodl-tmp/new_self_play_drpo"
if LOCAL_TRL_PARENT not in sys.path:
    sys.path.insert(0, LOCAL_TRL_PARENT)

In [ ]:
# ═══════════════════════════════════════════
# Pre-process dataset: arrange chosen/rejected from rank
# Push to august66/hh_helpfulness_qwen2.5_1.5b_generation_dpo
# ═══════════════════════════════════════════
from datasets import load_dataset

DATA_CACHE_DIR = "/root/autodl-tmp/data_cache"
SRC_DATASET = "august66/hh_helpfulness_qwen2.5_1.5b_generation"
DPO_DATASET = "august66/hh_helpfulness_qwen2.5_1.5b_generation_dpo"

raw = load_dataset(SRC_DATASET, split="train", cache_dir=DATA_CACHE_DIR)
print(f"Loaded {len(raw)} rows | Columns: {raw.column_names}")
print(raw[0])

def to_dpo_format(ex):
    """rank=1 means response_1 is preferred (chosen)."""
    if ex["rank"] == 1:
        chosen, rejected = ex["response_1"], ex["response_2"]
    else:
        chosen, rejected = ex["response_2"], ex["response_1"]
    return {"prompt": ex["prompt"], "chosen": chosen, "rejected": rejected}

dpo_ds = raw.map(to_dpo_format, remove_columns=[
    c for c in raw.column_names if c not in {"prompt", "chosen", "rejected"}
])
print(f"\nDPO dataset: {len(dpo_ds)} rows | Columns: {dpo_ds.column_names}")
print(f"\nSample:")
print(f"  prompt:   {str(dpo_ds[0]['prompt'])[:150]}")
print(f"  chosen:   {str(dpo_ds[0]['chosen'])[:150]}")
print(f"  rejected: {str(dpo_ds[0]['rejected'])[:150]}")

dpo_ds.push_to_hub(DPO_DATASET)
print(f"\nPushed to {DPO_DATASET}")

In [ ]:
# ═══════════════════════════════════════════
# DPO Training
# ═══════════════════════════════════════════
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig, ModelConfig

MODEL_CACHE_DIR = "/root/autodl-tmp/model_cache"
DATA_CACHE_DIR  = "/root/autodl-tmp/data_cache"
REF_MODEL       = "august66/qwen2.5-1.5b-base-hh-helpful-sft"
DPO_DATASET     = "august66/hh_helpfulness_qwen2.5_1.5b_generation_dpo"

def load_model(model_path):
    model_args = ModelConfig(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        trust_remote_code=model_args.trust_remote_code,
        cache_dir=MODEL_CACHE_DIR,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        padding_side="left",
        truncation_side="left",
        use_fast=True,
        trust_remote_code=model_args.trust_remote_code,
        cache_dir=MODEL_CACHE_DIR,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer

# Load models (same ref for both model and ref_model)
print("Loading model...")
model, tokenizer = load_model(REF_MODEL)
print("Loading ref model...")
ref_model, _ = load_model(REF_MODEL)
print("Models loaded.")

# Load DPO dataset
train_ds = load_dataset(DPO_DATASET, split="train", cache_dir=DATA_CACHE_DIR)
print(f"Training data: {len(train_ds)} rows")

# DPO config
config = DPOConfig(
    beta=0.1,
    learning_rate=5e-7,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    report_to="wandb",
    run_name="dpo-hh-qwen1.5b-from-generation",
    logging_steps=20,
    max_prompt_length=512,
    max_completion_length=512,
    max_length=1024,
    output_dir="dpo_out_generation",
    save_strategy="steps",
    save_steps=10000,
    save_total_limit=1,
    push_to_hub=True,
    hub_model_id="august66/hh_qwen_1.5b_sft_dpo_generation",
    hub_strategy="every_save",
    bf16=True,
    fp16=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()
print("DPO training complete.")